# AI Stock Price Prediction


## 1. Imports & Configuration

In [4]:
DOWNLOAD_DATA = False

In [5]:
%pip install akshare pandas numpy matplotlib scikit-learn xgboost

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Data Exploration

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import akshare as ak

if DOWNLOAD_DATA:
    df = ak.stock_zh_a_hist(
        symbol="000001",
        period="daily",
        start_date="20100101",
        end_date="20260913",
        adjust="qfq",
        timeout=30
    )

    df.to_csv("../data/raw/000001.csv", index=False)

else:
    df = pd.read_csv(
        "../data/raw/000001.csv"
    )

df.head()

,日期,股票代码,开盘,收盘,最高,最低,成交量,成交额,振幅,涨跌幅,涨跌额,换手率
0,2010-01-04,1,5.16,4.86,5.18,4.85,241923,5.802495e+08,6.47,-4.71,-0.24,0.83
1,2010-01-05,1,4.88,4.72,4.93,4.52,556500,1.293477e+09,8.44,-2.88,-0.14,1.90
2,2010-01-06,1,4.70,4.57,4.70,4.51,412143,9.444537e+08,4.03,-3.18,-0.15,1.41
3,2010-01-07,1,4.57,4.48,4.63,4.39,355337,8.041663e+08,5.25,-1.97,-0.09,1.22
4,2010-01-08,1,4.43,4.46,4.52,4.37,288543,6.506674e+08,3.35,-0.45,-0.02,0.99


In [7]:
from pathlib import Path

RAW_DATA_DIR = Path("../data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

raw_file = RAW_DATA_DIR / "000001.csv"

df.to_csv(raw_file, index=False)

print(f"Saved to: {raw_file}")

Saved to: ..\data\raw\000001.csv


In [8]:
print(df.shape)
print(df.columns.tolist())

(3985, 12)
['日期', '股票代码', '开盘', '收盘', '最高', '最低', '成交量', '成交额', '振幅', '涨跌幅', '涨跌额', '换手率']


In [9]:
display(df.head())
display(df.tail())

,日期,股票代码,开盘,收盘,最高,最低,成交量,成交额,振幅,涨跌幅,涨跌额,换手率
0,2010-01-04,1,5.16,4.86,5.18,4.85,241923,5.802495e+08,6.47,-4.71,-0.24,0.83
1,2010-01-05,1,4.88,4.72,4.93,4.52,556500,1.293477e+09,8.44,-2.88,-0.14,1.90
2,2010-01-06,1,4.70,4.57,4.70,4.51,412143,9.444537e+08,4.03,-3.18,-0.15,1.41
3,2010-01-07,1,4.57,4.48,4.63,4.39,355337,8.041663e+08,5.25,-1.97,-0.09,1.22
4,2010-01-08,1,4.43,4.46,4.52,4.37,288543,6.506674e+08,3.35,-0.45,-0.02,0.99


,日期,股票代码,开盘,收盘,最高,最低,成交量,成交额,振幅,涨跌幅,涨跌额,换手率
3980,2026-09-07,1,11.87,11.70,11.88,11.65,1087276,1.275606e+09,1.93,-1.60,-0.19,0.56
3981,2026-09-08,1,11.66,11.78,11.81,11.65,740516,8.703611e+08,1.37,0.68,0.08,0.38
3982,2026-09-09,1,11.76,11.70,11.79,11.69,582306,6.829468e+08,0.85,-0.68,-0.08,0.30
3983,2026-09-10,1,11.68,11.85,11.86,11.66,867632,1.022544e+09,1.71,1.28,0.15,0.45
3984,2026-09-11,1,11.82,11.74,11.86,11.71,832461,9.799906e+08,1.27,-0.93,-0.11,0.43


## 3. Feature Engineering

In [10]:
df = df.rename(columns={
    "日期": "date",
    "股票代码": "ticker",
    "开盘": "open",
    "收盘": "close",
    "最高": "high",
    "最低": "low",
    "成交量": "volume",
    "成交额": "amount",
    "振幅": "amplitude",
    "涨跌幅": "pct_change",
    "涨跌额": "price_change",
    "换手率": "turnover"
})

df["date"] = pd.to_datetime(df["date"])

df = (
    df
    .sort_values("date")
    .reset_index(drop=True)
)

df.head()

,date,ticker,open,close,high,low,volume,amount,amplitude,pct_change,price_change,turnover
0,2010-01-04,1,5.16,4.86,5.18,4.85,241923,5.802495e+08,6.47,-4.71,-0.24,0.83
1,2010-01-05,1,4.88,4.72,4.93,4.52,556500,1.293477e+09,8.44,-2.88,-0.14,1.90
2,2010-01-06,1,4.70,4.57,4.70,4.51,412143,9.444537e+08,4.03,-3.18,-0.15,1.41
3,2010-01-07,1,4.57,4.48,4.63,4.39,355337,8.041663e+08,5.25,-1.97,-0.09,1.22
4,2010-01-08,1,4.43,4.46,4.52,4.37,288543,6.506674e+08,3.35,-0.45,-0.02,0.99


In [11]:
# pct_change: (current_close - previous_close) / previous_close
df["return_1d"] = df["close"].pct_change(1)
df["return_3d"] = df["close"].pct_change(3)
df["return_5d"] = df["close"].pct_change(5)

# Calculate the n-day moving average 
#`rolling(5)` means: Creating a sliding window of length 5.
df["ma_5"] = df["close"].rolling(5).mean() #MA5: Average price over the last 5 days
df["ma_10"] = df["close"].rolling(10).mean()
df["ma_20"] = df["close"].rolling(20).mean()

#Measure how far the current price is from the 5-day moving average
df["ma5_distance"] = ( 
    df["close"] / df["ma_5"] - 1
)

df["ma20_distance"] = (
    df["close"] / df["ma_20"] - 1
)

#Calculate 5-day rolling volatility using daily returns
df["volatility_5d"] = (
    df["return_1d"]
    .rolling(5)
    .std()
)

df["volatility_20d"] = (
    df["return_1d"]
    .rolling(20)
    .std()
)

# Calculate the daily percentage change in trading volume
df["volume_change"] = df["volume"].pct_change()


# Calculate the next trading day's return
df["next_day_return"] = (
    df["close"].shift(-1) / df["close"] - 1
)

# Create the binary target:
# 1 = next trading day goes up
# 0 = next trading day does not go up
df["target"] = (
    df["next_day_return"] > 0
).astype(int)

In [12]:
# Define the feature columns used for model training
feature_columns = [
    "return_1d",
    "return_3d",
    "return_5d",
    "ma5_distance",
    "ma20_distance",
    "volatility_5d",
    "volatility_20d",
    "volume_change"
]


# Display the engineered features for inspection
df[
    ["date", "close"] + feature_columns
].head(25)

,date,close,return_1d,return_3d,return_5d,ma5_distance,ma20_distance,volatility_5d,volatility_20d,volume_change
0,2010-01-04,4.86,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-05,4.72,-0.028807,NaN,NaN,NaN,NaN,NaN,NaN,1.300319
2,2010-01-06,4.57,-0.031780,NaN,NaN,NaN,NaN,NaN,NaN,-0.259402
3,2010-01-07,4.48,-0.019694,-0.078189,NaN,NaN,NaN,NaN,NaN,-0.137831
4,2010-01-08,4.46,-0.004464,-0.055085,NaN,-0.034214,NaN,NaN,NaN,-0.187974
5,2010-01-11,4.46,0.000000,-0.024070,-0.082305,-0.017188,NaN,0.014241,NaN,0.534766
6,2010-01-12,4.41,-0.011211,-0.015625,-0.065678,-0.014745,NaN,0.012665,NaN,0.336347
7,2010-01-13,3.87,-0.122449,-0.132287,-0.153173,-0.107472,NaN,0.051347,NaN,0.580002
8,2010-01-14,3.87,0.000000,-0.132287,-0.136161,-0.081633,NaN,0.053206,NaN,-0.442595
9,2010-01-15,4.04,0.043928,-0.083900,-0.094170,-0.021792,NaN,0.062126,NaN,0.035137


In [13]:
# Select only the columns required for machine learning
model_df = df[
    ["date"] + feature_columns +
    ["next_day_return", "target"]
].copy()

# Replace infinite values with NaN
model_df = model_df.replace(
    [np.inf, -np.inf],
    np.nan
)

# Remove rows containing missing values(NaN)
model_df = model_df.dropna()

# Reset the DataFrame index
model_df = model_df.reset_index(drop=True)

# Compare the number of rows before and after cleaning
print("Original rows:", len(df))
print("Model rows:", len(model_df))
 
model_df.head()

Original rows: 3985
Model rows: 3964


,date,return_1d,return_3d,return_5d,ma5_distance,ma20_distance,volatility_5d,volatility_20d,volume_change,next_day_return,target
0,2010-02-01,-0.041063,-0.057007,-0.078886,-0.042912,-0.070910,0.014210,0.051103,0.176544,0.012594,1
1,2010-02-02,0.012594,-0.033654,-0.056338,-0.019512,-0.051439,0.019361,0.051089,0.012550,0.109453,1
2,2010-02-03,0.109453,0.077295,0.059382,0.074699,0.053751,0.057363,0.056880,0.914687,-0.031390,0
3,2010-02-04,-0.031390,0.088161,0.038462,0.032999,0.022606,0.060067,0.057156,-0.447463,-0.016204,0
4,2010-02-05,-0.016204,0.057214,0.026570,0.010942,0.008543,0.060929,0.057261,-0.089574,-0.037647,0


## 4. Train / Validation / Test Split

In [14]:
X = model_df[feature_columns]

y = model_df["target"]

In [15]:
# Calculate the split positions based on chronological order
train_end = int(len(model_df) * 0.70)
val_end = int(len(model_df) * 0.85)
 
# Split the features chronologically into training, validation, and test sets
X_train = X.iloc[:train_end]
X_val = X.iloc[train_end:val_end]
X_test = X.iloc[val_end:]

# Split the target chronologically into training, validation, and test sets
y_train = y.iloc[:train_end]
y_val = y.iloc[train_end:val_end]
y_test = y.iloc[val_end:]

# backtest 回测  ：Use historical data, pretend that you didn't know about the future at that time, and then test how your trading strategy performed in the past.
dates_train = model_df["date"].iloc[:train_end]
dates_val = model_df["date"].iloc[train_end:val_end]
dates_test = model_df["date"].iloc[val_end:]
 
print("Train samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))

# Display the date range of each dataset
print(
    "\nTrain:",
    dates_train.min(),
    "->",
    dates_train.max()
)

print(
    "Validation:",
    dates_val.min(),
    "->",
    dates_val.max()
)

print(
    "Test:",
    dates_test.min(),
    "->",
    dates_test.max()
)

Train samples: 2774
Validation samples: 595
Test samples: 595

Train: 2010-02-01 00:00:00 -> 2021-10-19 00:00:00
Validation: 2021-10-20 00:00:00 -> 2024-03-29 00:00:00
Test: 2024-04-01 00:00:00 -> 2026-09-10 00:00:00


## 5. Logistic Regression Baseline

### 5.1 Model Setup and Training

In [16]:
 
from sklearn.linear_model import LogisticRegression 
from sklearn.preprocessing import StandardScaler 
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

#Because the scales of features may be different
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

_ =logistic_model.fit(
    X_train_scaled,
    y_train
)

### 5.2 Train and Validation Accuracy

In [17]:
# Predict on training and validation sets
y_train_pred = logistic_model.predict(X_train_scaled)
y_val_pred = logistic_model.predict(X_val_scaled)

# Predict probability of the positive class
y_val_prob = logistic_model.predict_proba(X_val_scaled)[:, 1]

# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Train Accuracy     : {train_accuracy:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

Train Accuracy     : 0.5292
Validation Accuracy: 0.5496


### 5.3 Validation Metrics

In [18]:
val_precision = precision_score(
    y_val,
    y_val_pred,
    zero_division=0
)

val_recall = recall_score(
    y_val,
    y_val_pred,
    zero_division=0
)

val_f1 = f1_score(
    y_val,
    y_val_pred,
    zero_division=0
)

#  Accuracy 很受 threshold 影响。而 AUC 更关心：正样本的 probability 是不是通常比负样本高
# AUC: positive sample 的 score 应该比 negative sample 高
val_auc = roc_auc_score(
    y_val,
    y_val_prob
)

print(f"Precision: {val_precision:.4f}")
print(f"Recall   : {val_recall:.4f}")
print(f"F1 Score : {val_f1:.4f}")
print(f"ROC-AUC  : {val_auc:.4f}")

Precision: 0.4364
Recall   : 0.1890
F1 Score : 0.2637
ROC-AUC  : 0.5177


### 5.4 Class Distribution

In [19]:
print(y_val.value_counts())
print()

print(y_val.value_counts(normalize=True))

target
0    341
1    254
Name: count, dtype: int64

target
0    0.573109
1    0.426891
Name: proportion, dtype: float64


### 5.5 Confusion Matrix

In [20]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_val,
    y_val_pred
)

print(cm)

[[279  62]
 [206  48]]


### 5.6 Dummy Baseline Comparison

In [21]:
from sklearn.dummy import DummyClassifier

dummy_model = DummyClassifier(
    strategy="most_frequent" # Always predict the category that appears most frequently in the training set.
)

dummy_model.fit(
    X_train_scaled,
    y_train
)

dummy_pred = dummy_model.predict(
    X_val_scaled
)

dummy_accuracy = accuracy_score(
    y_val,
    dummy_pred
)

print(f"Dummy Accuracy   : {dummy_accuracy:.4f}")
print(f"Logistic Accuracy: {val_accuracy:.4f}")

Dummy Accuracy   : 0.5731
Logistic Accuracy: 0.5496


In [22]:
print("Training class distribution:")
print(y_train.value_counts())
print()

print("Training class ratio:")
print(y_train.value_counts(normalize=True))

Training class distribution:
target
0    1450
1    1324
Name: count, dtype: int64

Training class ratio:
target
0    0.522711
1    0.477289
Name: proportion, dtype: float64


### 5.7 Observations

- The training dataset is relatively balanced, with approximately 52.3% class 0 and 47.7% class 1.
- Logistic Regression achieved a validation accuracy of 54.96%.
- The majority-class Dummy Classifier achieved a higher validation accuracy of 57.31%.
- The ROC-AUC score of 0.5177 is close to random guessing.
- These results suggest that the current Logistic Regression model has very limited predictive power for next-day stock direction.
- More complex non-linear models will be evaluated next.

## 6. Random Forest


### 6.1 Model Training

In [23]:
from sklearn.ensemble import RandomForestClassifier

# Create the Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

# Train the model
rf_model.fit(
    X_train,
    y_train
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total n

### 6.2 Train and Validation Accuracy

In [24]:
# Predict on training and validation sets
y_train_pred_rf = rf_model.predict(X_train)
y_val_pred_rf = rf_model.predict(X_val)

# Calculate accuracy
train_accuracy_rf = accuracy_score(
    y_train,
    y_train_pred_rf
)

val_accuracy_rf = accuracy_score(
    y_val,
    y_val_pred_rf
)

print(f"Train Accuracy     : {train_accuracy_rf:.4f}")
print(f"Validation Accuracy: {val_accuracy_rf:.4f}")

Train Accuracy     : 0.6828
Validation Accuracy: 0.5697


### 6.3 Validation Metrics

In [25]:
# Predict probability of class 1
y_val_prob_rf = rf_model.predict_proba(
    X_val
)[:, 1]

# Calculate validation metrics
val_precision_rf = precision_score(
    y_val,
    y_val_pred_rf,
    zero_division=0
)

val_recall_rf = recall_score(
    y_val,
    y_val_pred_rf,
    zero_division=0
)

val_f1_rf = f1_score(
    y_val,
    y_val_pred_rf,
    zero_division=0
)

val_auc_rf = roc_auc_score(
    y_val,
    y_val_prob_rf
)

print(f"Precision: {val_precision_rf:.4f}")
print(f"Recall   : {val_recall_rf:.4f}")
print(f"F1 Score : {val_f1_rf:.4f}")
print(f"ROC-AUC  : {val_auc_rf:.4f}")

Precision: 0.4932
Recall   : 0.2835
F1 Score : 0.3600
ROC-AUC  : 0.5456


### 6.4 Classification Report

               所有预测结果  
                    ↓  
          ┌─────────┴─────────┐  
          ↓                   ↓  
      整体 Accuracy       分类别评估  
                              ↓  
                     class 0 precision/recall/F1  
                     class 1 precision/recall/F1  
                              ↓  
                     macro / weighted average  

In [26]:
print(
    classification_report(
        y_val,
        y_val_pred_rf,
        digits=4,
        zero_division=0
    )
)

              precision    recall  f1-score   support

           0     0.5947    0.7830    0.6759       341
           1     0.4932    0.2835    0.3600       254

    accuracy                         0.5697       595
   macro avg     0.5439    0.5332    0.5180       595
weighted avg     0.5513    0.5697    0.5411       595



### 6.5 Feature Importance

In [ ]:
# Sklearn will calculate:  This split has reduced the impurity by how much.

# If a certain feature is frequently used for splitting and each time it can significantly purify the node, then its importance will be relatively high.

feature_importance_rf = pd.DataFrame({
    "feature": feature_columns,
    "importance": rf_model.feature_importances_
})

feature_importance_rf = (
    feature_importance_rf
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

feature_importance_rf

,feature,importance
0,return_1d,0.146304
1,ma20_distance,0.137480
2,return_3d,0.132903
3,ma5_distance,0.129311
4,volume_change,0.118659
5,volatility_20d,0.116017
6,return_5d,0.113442
7,volatility_5d,0.105882


### 6.6 Observations

- Random Forest achieved a training accuracy of 68.28% and a validation accuracy of 56.97%, indicating some overfitting.
- The validation ROC-AUC improved to 0.5456, which is better than Logistic Regression (0.5177), but the predictive signal is still weak.
- Recall for class 1 is only 28.35%, meaning the model still misses many actual upward movements.
- The validation accuracy of 56.97% is still slightly lower than the Dummy Classifier baseline of 57.31%.
- `return_1d`, `ma20_distance`, and `return_3d` were the most important features in the current Random Forest model.

## 7. XGBoost


### 7.1 Model Training

In [ ]:
from xgboost import XGBClassifier

# Create the XGBoost model
xgb_model = XGBClassifier(
    n_estimators=200, #最多建立 200 棵 boosting tree
    max_depth=4,
    learning_rate=0.05, #每棵新树只做“小幅修正  
    subsample=0.8,#每棵树随机使用 80% 的训练样本 ->随机抽样本
    colsample_bytree=0.8,#每棵树随机使用 80% 的 features ->随机抽特征
    random_state=42,
    eval_metric="logloss"
)
# these settings are used to reduce overfitting

# Train the model
xgb_model.fit(
    X_train,
    y_train
)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


### 7.2 Train and Validation Accuracy

In [29]:
y_train_pred_xgb = xgb_model.predict(X_train)
y_val_pred_xgb = xgb_model.predict(X_val)

train_accuracy_xgb = accuracy_score(
    y_train,
    y_train_pred_xgb
)

val_accuracy_xgb = accuracy_score(
    y_val,
    y_val_pred_xgb
)

print(f"Train Accuracy     : {train_accuracy_xgb:.4f}")
print(f"Validation Accuracy: {val_accuracy_xgb:.4f}")

Train Accuracy     : 0.8039
Validation Accuracy: 0.5580


### 7.3 Validation Metrics

In [30]:
y_val_prob_xgb = xgb_model.predict_proba(X_val)[:, 1]

val_precision_xgb = precision_score(
    y_val,
    y_val_pred_xgb,
    zero_division=0
)

val_recall_xgb = recall_score(
    y_val,
    y_val_pred_xgb,
    zero_division=0
)

val_f1_xgb = f1_score(
    y_val,
    y_val_pred_xgb,
    zero_division=0
)

val_auc_xgb = roc_auc_score(
    y_val,
    y_val_prob_xgb
)

print(f"Precision: {val_precision_xgb:.4f}")
print(f"Recall   : {val_recall_xgb:.4f}")
print(f"F1 Score : {val_f1_xgb:.4f}")
print(f"ROC-AUC  : {val_auc_xgb:.4f}")

Precision: 0.4829
Recall   : 0.5000
F1 Score : 0.4913
ROC-AUC  : 0.5440


### 7.4 Classification Report

In [31]:
print(
    classification_report(
        y_val,
        y_val_pred_xgb,
        digits=4,
        zero_division=0
    )
)

              precision    recall  f1-score   support

           0     0.6175    0.6012    0.6092       341
           1     0.4829    0.5000    0.4913       254

    accuracy                         0.5580       595
   macro avg     0.5502    0.5506    0.5503       595
weighted avg     0.5600    0.5580    0.5589       595



### 7.5 Feature Importance

In [32]:
feature_importance_xgb = pd.DataFrame({
    "feature": feature_columns,
    "importance": xgb_model.feature_importances_
})

feature_importance_xgb = (
    feature_importance_xgb
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

feature_importance_xgb

,feature,importance
0,return_3d,0.132048
1,ma20_distance,0.130945
2,volatility_20d,0.126264
3,volume_change,0.125049
4,ma5_distance,0.124507
5,return_1d,0.122200
6,return_5d,0.120613
7,volatility_5d,0.118374


### 7.6 Observations

- XGBoost achieved a high training accuracy of 80.39%, but the validation accuracy was only 55.80%, indicating clear overfitting.
- The validation ROC-AUC was 0.5440, which is slightly lower than the Random Forest result of 0.5456.
- XGBoost achieved a higher recall for class 1 (50.00%) than Random Forest, meaning it identified more actual upward movements.
- However, its precision for class 1 was only 48.29%, so many predicted upward movements were incorrect.
- The current XGBoost model does not outperform Random Forest on the validation set.
- Feature importance was relatively evenly distributed, with `return_3d`, `ma20_distance`, and `volatility_20d` receiving the highest importance scores.

## 8. Model Comparison

### 8.1 Validation Performance Comparison

In [33]:
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "Train Accuracy": [
        train_accuracy,
        train_accuracy_rf,
        train_accuracy_xgb
    ],
    "Validation Accuracy": [
        val_accuracy,
        val_accuracy_rf,
        val_accuracy_xgb
    ],
    "F1 Score": [
        val_f1,
        val_f1_rf,
        val_f1_xgb
    ],
    "ROC-AUC": [
        val_auc,
        val_auc_rf,
        val_auc_xgb
    ]
})

model_comparison

,Model,Train Accuracy,Validation Accuracy,F1 Score,ROC-AUC
0,Logistic Regression,0.529200,0.549580,0.263736,0.517734
1,Random Forest,0.682769,0.569748,0.360000,0.545628
2,XGBoost,0.803893,0.557983,0.491296,0.543965


## 9. XGBoost Tuning

### 9.1 Reduce Overfitting

In [ ]:
xgb_model_tuned = XGBClassifier(
    n_estimators=150,
    max_depth=2, # reduce depth
    learning_rate=0.03,
    min_child_weight=5, #If a node has too few samples/weights, then stop further division.
    subsample=0.7,# 
    colsample_bytree=0.7,# 
    reg_alpha=0.5, # Imposing a penalty on the complexity of the mode
    reg_lambda=2.0,
    random_state=42,
    eval_metric="logloss"
)

xgb_model_tuned.fit(
    X_train,
    y_train
)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.7
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


### 9.2 Tuned Model Evaluation

In [37]:
y_train_pred_xgb_tuned = xgb_model_tuned.predict(
    X_train
)

y_val_pred_xgb_tuned = xgb_model_tuned.predict(
    X_val
)

y_val_prob_xgb_tuned = xgb_model_tuned.predict_proba(
    X_val
)[:, 1]

train_accuracy_xgb_tuned = accuracy_score(
    y_train,
    y_train_pred_xgb_tuned
)

val_accuracy_xgb_tuned = accuracy_score(
    y_val,
    y_val_pred_xgb_tuned
)

val_precision_xgb_tuned = precision_score(
    y_val,
    y_val_pred_xgb_tuned,
    zero_division=0
)

val_recall_xgb_tuned = recall_score(
    y_val,
    y_val_pred_xgb_tuned,
    zero_division=0
)

val_f1_xgb_tuned = f1_score(
    y_val,
    y_val_pred_xgb_tuned,
    zero_division=0
)

val_auc_xgb_tuned = roc_auc_score(
    y_val,
    y_val_prob_xgb_tuned
)

print(f"Train Accuracy     : {train_accuracy_xgb_tuned:.4f}")
print(f"Validation Accuracy: {val_accuracy_xgb_tuned:.4f}")
print(f"Precision          : {val_precision_xgb_tuned:.4f}")
print(f"Recall             : {val_recall_xgb_tuned:.4f}")
print(f"F1 Score           : {val_f1_xgb_tuned:.4f}")
print(f"ROC-AUC            : {val_auc_xgb_tuned:.4f}")

Train Accuracy     : 0.6045
Validation Accuracy: 0.5697
Precision          : 0.4944
Recall             : 0.3465
F1 Score           : 0.4074
ROC-AUC            : 0.5374


### 9.3 Observations

- Regularization substantially reduced XGBoost overfitting.
- Training accuracy decreased from 80.39% to 60.45%, while validation accuracy improved slightly from 55.80% to 56.97%.
- The training-validation gap became much smaller, indicating better generalization.
- However, ROC-AUC decreased from 0.5440 to 0.5374, and the F1 score also decreased.
- The tuned XGBoost model is more stable, but it does not outperform Random Forest on validation performance.

# To do list
## 10. Extra Trees
## 11. LightGBM 
## 12. Model Comparison 
    Accuracy
    Precision
    Recall
    F1
    ROC-AUC
    Overfitting gap

## 13. Feature Selection
    13.1 Feature Correlation
    13.2 Random Forest / XGBoost Importance
    13.3 GA-based Feature Selection
    13.4 Compare All Features vs Selected Features

## 14. Hyperparameter Tuning  
## 15. Final Model Selection  
## 16. Final Test Evaluation 
## 17. Prediction Confidence
## 18. Backtesting
## 19. Explainability
## 20. Save Model